In [1]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['CRYPTOGRAPHY_OPENSSL_NO_LEGACY'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

import datasets
from datasets import load_dataset, Dataset, DatasetDict, Dataset
from tqdm.auto import tqdm
import json

datasets.disable_caching()

/inspire/hdd/global_user/zhangxiaofan-25025/genggui001/miniconda3/envs/megatron-next/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from megatron.energon import get_train_dataset, WorkerConfig, get_loader, DefaultTaskEncoder, Cooker, basic_sample_keys
from megatron.energon import DefaultTaskEncoder, TextSample, Sample
import gzip
import pickle
import webdataset as wds
import json
import dataclasses
import torch

from tqdm.auto import tqdm
from typing import List

from qwen3_base_energon_helpers import MyTaskEncoder, print_error_handler

In [3]:
tokenizer_path="/inspire/hdd/global_user/zhangxiaofan-25025/genggui001/code/Megatron-Next/model_dir/pulse_v19_1_80b_a3b_next_gemini_bf16/tokenizer_v19_gemini"

In [4]:
worker_config = WorkerConfig(
    rank=0,
    world_size=1,
    seed_offset=123456789,
    num_workers=1,
)

train_ds = get_train_dataset(
    '/inspire/hdd/project/qproject-medicineresearch/public/inspire_shared/mount/genggui001_pa/dataset/jarvis_corpus/en_and_code/gg_en_med/ns/data',
    split_part="train",
    task_encoder=MyTaskEncoder(
        tokenizer_path=tokenizer_path,
        seq_length=66536,
        sensitive_words_path=None,
    ),
    batch_size=1,
    packing_buffer_size=None,
    shuffle_buffer_size=None,
    max_samples_per_sequence=None,
    worker_config=worker_config,
)

train_dataloader = get_loader(train_ds, worker_config=worker_config,)

rank=0, worker=0: sample_range=[0, 188666968] in 189 slices, sum(count)=188666968: indexes=[0, 1000000, 2000000 ...<184> 186666968, 187666968, 188666968] slices=[train-chunk-0.tar[0(start), 1000000(end)], train-chunk-1.tar[0(start), 1000000(end)], train-chunk-10.tar[0(start), 1000000(end)] ...<184> train-chunk-97.tar[0(start), 1000000(end)], train-chunk-98.tar[0(start), 1000000(end)], train-chunk-99.tar[0(start), 1000000(end)]]


/inspire/hdd/global_user/zhangxiaofan-25025/genggui001/miniconda3/envs/megatron-next/lib/python3.12/site-packages/megatron/energon/loader.py:108: FutureWarning: Passing a worker_config to get_loader() is deprecated and will have no effect.
  warn_deprecated(


In [5]:
dd = iter(train_dataloader)

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

In [7]:
import json

count_map = {}

with open("/inspire/hdd/global_user/zhangxiaofan-25025/genggui001/code/Megatron-Next/model_dir/pulse_v19_1_80b_a3b_next_gemini_bf16/qua/gg_en_med.jsonl", "w", encoding="utf8") as f:
    for _ in tqdm(range(2048)):
        item = next(dd)

              
        item = {
            "text": tokenizer.decode([item for item in item['input_ids'].tolist()[0] if item != -100])
        }
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        

            
        

100%|██████████| 2048/2048 [00:11<00:00, 178.23it/s]


Token indices sequence length is longer than the specified maximum sequence length for this model (144130 > 66536). Running this sequence through the model will result in indexing errors


In [ ]:
count_list = sorted(count_map.items(), key=lambda x:-x[1])
[(k,v/2048)for k,v in count_list]

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

In [ ]:
tokenizer("<|im_start|>assistant\n<think>\n").input_ids 

In [ ]:
tokenizer("<|im_start|>assistant\n<ggthink>\n").input_ids 

In [44]:
item = next(dd)
# print(item['__key__'][0].split("/")[-1].split("-train-")[0])
print(tokenizer.decode([item for item in item['input_ids'].tolist()[0] if item != -100]))

面对幽门螺杆菌,这些常见问题你一定要懂

首先我们先要认识一下这个幽门螺杆菌,目前专家一致认为生长在胃黏膜内的幽门螺杆菌,是一种传染病菌,可在人与人之间传播,现已明确与多种疾病的发病有关。专家认为对有下列临床特征的人群进行筛查,对阳性者进行根除治疗并能从中获益:
(1)胃痛胀等消化不良症状,对症治疗无效;
(2)糜烂、出血、萎缩性胃炎、异型增生、肠化;
(3)胃十二指肠溃疡,无论有否症状或并发症;
(4)胃癌家族史、或胃癌切除手术后;
(5)长期服用质子泵抑制剂、非甾体类抗炎止痛药、低剂量阿司匹林等;
(6)胃MALT淋巴瘤、淋巴细胞性胃炎、增生性胃息肉、Menetrier病等;
(7)缺铁性贫血、特发性血小板减少性紫癜、维生素B12缺乏症等。


In [ ]:
print(tokenizer.decode([item for item in item['labels'].tolist()[0] if item != -100]))